# Big Bash League team performance: analysis notebook

This notebook walks through data quality first, then the main questions, then the conclusions.
It reads the files in `output/`, so run `python data_processing.py` before opening it.
All figures come from the same functions the dashboard uses.

## Part 1: data quality

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

OUT = Path.cwd().parent / "output"          # the notebook lives in notebooks/, output/ is one level up
sys.path.insert(0, str(Path.cwd().parent))  # so we can import data_processing.py
from data_processing import (MIN_DECIDED_RELIABLE, PHASE_LABELS, build_findings, phase_table,
                             season_order, team_kpis)

print((OUT / "data_quality_report.md").read_text(encoding="utf-8"))

In [ ]:
summary = json.loads((OUT / "cleaning_summary.json").read_text())
for table, info in summary["tables"].items():
    print(f"{table}: {info['raw_rows']:,} raw rows, {info['clean_rows']:,} clean rows, "
          f"{info['duplicates_removed']} duplicates removed")
print("Outliers flagged (kept, not deleted):", summary["outliers"]["flagged_rows"])

## Part 2: the questions

Each section answers one question and shows the table or chart behind the answer.

In [ ]:
matches = pd.read_csv(OUT / "cleaned_matches.csv", dtype={"season": str}, parse_dates=["date"])
balls = pd.read_csv(OUT / "cleaned_ball_by_ball.csv", dtype={"season": str}, low_memory=False)
kpis = team_kpis(matches, balls)
season_list = season_order(matches)
print(f"{len(matches):,} matches, {len(balls):,} deliveries, {len(kpis)} teams, "
      f"{matches['date'].min().date()} to {matches['date'].max().date()}")

### Which team wins the largest share of its decided matches?

In [ ]:
kpis[["matches_played", "decided_matches", "wins", "win_rate_pct"]].sort_values("win_rate_pct", ascending=False)

In [ ]:
k = kpis.sort_values("win_rate_pct")
colors = ["#d9822b" if n >= MIN_DECIDED_RELIABLE else "#9e9e9e" for n in k["decided_matches"]]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(k.index, k["win_rate_pct"], color=colors)
ax.set_xlabel("Win rate (% of decided matches)")
ax.set_title(f"Grey = fewer than {MIN_DECIDED_RELIABLE} decided matches")
plt.tight_layout()
plt.show()

### Does winning the toss help?

In [ ]:
decided = matches[matches["winner"].notna()]
overall = (decided["toss_winner"] == decided["winner"]).mean() * 100
print(f"Across all {len(decided):,} decided matches, the toss winner won {overall:.1f}% of the time.")

by_choice = (decided.assign(toss_winner_won=decided["toss_winner"] == decided["winner"])
             .groupby("toss_decision")["toss_winner_won"].agg(["mean", "size"]))
by_choice["mean"] = (by_choice["mean"] * 100).round(1)
by_choice.rename(columns={"mean": "toss winner won (%)", "size": "decided matches"})

In [ ]:
kpis[["decided_after_toss_win", "win_rate_after_toss_win_pct",
      "decided_after_toss_loss", "win_rate_after_toss_loss_pct"]]

### How fast do teams score in each phase, season by season?

In [ ]:
pt = phase_table(balls)
rates = (pt.groupby(["season", "phase"])[["runs", "legal_balls"]].sum()
         .assign(rate=lambda d: 100 * d["runs"] / d["legal_balls"]).reset_index())
wide = rates.pivot(index="season", columns="phase", values="rate").reindex(season_list)[PHASE_LABELS].round(1)
wide

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for phase in PHASE_LABELS:
    ax.plot(wide.index, wide[phase].to_numpy(), marker="o", label=phase)
ax.set_ylabel("Runs per 100 balls")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

### How are batters dismissed?

In [ ]:
d = balls[balls["counts_as_dismissal"] == 1]["wicket_kind"].value_counts(normalize=True).mul(100).round(1)
d.rename("share of dismissals (%)").to_frame()

## Part 3: conclusions

These sentences are written by `build_findings()` in `data_processing.py` from the data above,
so they always match the tables. Caveats are included.

In [ ]:
for heading, text in build_findings(kpis, matches, balls):
    display(Markdown(f"**{heading}.** {text}"))